In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, cross_validate
import joblib

In [4]:
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve
)


In [5]:
df=pd.read_csv('data.csv')



In [6]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200020 entries, 0 to 200019
Data columns (total 17 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   Patient ID                200020 non-null  int64  
 1   Heart Rate                200020 non-null  int64  
 2   Respiratory Rate          200020 non-null  int64  
 3   Timestamp                 200020 non-null  object 
 4   Body Temperature          200020 non-null  float64
 5   Oxygen Saturation         200020 non-null  float64
 6   Systolic Blood Pressure   200020 non-null  int64  
 7   Diastolic Blood Pressure  200020 non-null  int64  
 8   Age                       200020 non-null  int64  
 9   Gender                    200020 non-null  object 
 10  Weight (kg)               200020 non-null  float64
 11  Height (m)                200020 non-null  float64
 12  Derived_HRV               200020 non-null  float64
 13  Derived_Pulse_Pressure    200020 non-null  i

In [6]:
df.describe()



,Patient ID,Heart Rate,Respiratory Rate,Body Temperature,Oxygen Saturation,Systolic Blood Pressure,Diastolic Blood Pressure,Age,Weight (kg),Height (m),Derived_HRV,Derived_Pulse_Pressure,Derived_BMI,Derived_MAP
count,200020.000000,200020.000000,200020.000000,200020.000000,200020.000000,200020.000000,200020.000000,200020.000000,200020.000000,200020.000000,200020.000000,200020.000000,200020.000000,200020.000000
mean,100010.500000,79.533747,15.489451,36.748353,97.504372,124.437971,79.499625,53.446275,74.996419,1.750031,0.099970,44.938346,25.003625,94.479074
std,57740.944759,11.552894,2.294472,0.433290,1.442598,8.656946,5.757248,20.786802,14.471502,0.144554,0.028861,10.404945,6.447143,4.797891
min,1.000000,60.000000,12.000000,36.000004,95.000007,110.000000,70.000000,18.000000,50.000156,1.500001,0.050000,21.000000,12.505974,83.333333
25%,50005.750000,70.000000,13.000000,36.372613,96.256859,117.000000,75.000000,35.000000,62.423615,1.624777,0.074955,37.000000,20.134367,91.000000
50%,100010.500000,80.000000,15.000000,36.747741,97.509629,124.000000,79.000000,53.000000,74.977169,1.750478,0.099988,45.000000,24.320776,94.333333
75%,150015.250000,90.000000,17.000000,37.123003,98.755722,132.000000,84.000000,71.000000,87.539510,1.875310,0.124917,53.000000,29.187169,98.000000
max,200020.000000,99.000000,19.000000,37.499992,99.999963,139.000000,89.000000,89.000000,99.999765,1.999997,0.149999,69.000000,44.376487,105.666667


In [7]:
df.head()


,Patient ID,Heart Rate,Respiratory Rate,Timestamp,Body Temperature,Oxygen Saturation,Systolic Blood Pressure,Diastolic Blood Pressure,Age,Gender,Weight (kg),Height (m),Derived_HRV,Derived_Pulse_Pressure,Derived_BMI,Derived_MAP,Risk Category
0,1,60,12,2024-07-19 21:53:45.729841,36.861707,95.702046,124,86,37,Female,91.541618,1.679351,0.121033,38,32.459031,98.666667,High Risk
1,2,63,18,2024-07-19 21:52:45.729841,36.511633,96.689413,126,84,77,Male,50.704921,1.992546,0.117062,42,12.771246,98.000000,High Risk
2,3,63,15,2024-07-19 21:51:45.729841,37.052049,98.508265,131,78,68,Female,90.316760,1.770228,0.053200,53,28.821069,95.666667,Low Risk
3,4,99,16,2024-07-19 21:50:45.729841,36.654748,95.011801,118,72,41,Female,96.006188,1.833629,0.064475,46,28.554611,87.333333,High Risk
4,5,69,16,2024-07-19 21:49:45.729841,36.975098,98.623792,138,76,25,Female,56.020006,1.866419,0.118484,62,16.081438,96.666667,High Risk


In [8]:
df.isnull().sum()


Patient ID                  0
Heart Rate                  0
Respiratory Rate            0
Timestamp                   0
Body Temperature            0
Oxygen Saturation           0
Systolic Blood Pressure     0
Diastolic Blood Pressure    0
Age                         0
Gender                      0
Weight (kg)                 0
Height (m)                  0
Derived_HRV                 0
Derived_Pulse_Pressure      0
Derived_BMI                 0
Derived_MAP                 0
Risk Category               0
dtype: int64

In [9]:
df.duplicated().sum()
df.drop_duplicates(inplace=True)


In [10]:
for col in df.select_dtypes(include='number'):
    df[col].fillna(df[col].median(),inplace=True)


/tmp/ipykernel_24497/3630613446.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(),inplace=True)
/tmp/ipykernel_24497/3630613446.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try u

In [11]:
df.isnull().sum()


Patient ID                  0
Heart Rate                  0
Respiratory Rate            0
Timestamp                   0
Body Temperature            0
Oxygen Saturation           0
Systolic Blood Pressure     0
Diastolic Blood Pressure    0
Age                         0
Gender                      0
Weight (kg)                 0
Height (m)                  0
Derived_HRV                 0
Derived_Pulse_Pressure      0
Derived_BMI                 0
Derived_MAP                 0
Risk Category               0
dtype: int64

In [7]:
df['Risk Category']=df['Risk Category'].map({
    'High Risk':1,
    'Low Risk': 0
})


In [8]:
df["Gender"] = df["Gender"].map({
    "Male": 1,
    "Female": 0
})



In [9]:
df.shape


(200020, 17)

In [16]:
df.columns

Index(['Patient ID', 'Heart Rate', 'Respiratory Rate', 'Timestamp',
       'Body Temperature', 'Oxygen Saturation', 'Systolic Blood Pressure',
       'Diastolic Blood Pressure', 'Age', 'Gender', 'Weight (kg)',
       'Height (m)', 'Derived_HRV', 'Derived_Pulse_Pressure', 'Derived_BMI',
       'Derived_MAP', 'Risk Category'],
      dtype='object')

In [11]:
X = df.drop(['Risk Category','Timestamp','Patient ID'], axis=1)
y = df['Risk Category']


In [18]:
X.columns


Index(['Heart Rate', 'Respiratory Rate', 'Body Temperature',
       'Oxygen Saturation', 'Systolic Blood Pressure',
       'Diastolic Blood Pressure', 'Age', 'Gender', 'Weight (kg)',
       'Height (m)', 'Derived_HRV', 'Derived_Pulse_Pressure', 'Derived_BMI',
       'Derived_MAP'],
      dtype='object')

In [ ]:
df.shape


In [12]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        class_weight="balanced",
        random_state=42
    ))
])

pipeline.fit(X_train, y_train)



,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2


In [13]:
print(y_train.value_counts())
print(pipeline.named_steps['model'].classes_)



Risk Category
1    84092
0    75924
Name: count, dtype: int64
[0 1]


In [14]:
y_pred = pipeline.predict(X_test)  # Predicted classes
y_pred_proba = pipeline.predict_proba(X_test)[:, 1]


In [15]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")


Accuracy: 1.0000 (100.00%)


In [23]:
precision = precision_score(y_test, y_pred, pos_label=1)
print(f"Precision: {precision:.4f}")


Precision: 1.0000


In [24]:
recall = recall_score(y_test, y_pred, pos_label=1)
print(f"Recall: {recall:.4f}")


Recall: 1.0000


In [25]:
f1 = f1_score(y_test, y_pred, pos_label=1)
print(f"F1-Score: {f1:.4f}")


F1-Score: 1.0000


In [21]:
FEATURE_ORDER = [
    "Heart Rate",
    "Respiratory Rate",
    "Body Temperature",
    "Oxygen Saturation",
    "Systolic Blood Pressure",
    "Diastolic Blood Pressure",
    "Age",
    "Gender",
    "Weight (kg)",
    "Height (m)",
    "Derived_HRV",
    "Derived_Pulse_Pressure",
    "Derived_BMI",
    "Derived_MAP"
]



In [23]:
patient_input = {
    "Heart Rate": 150,               # Tachycardia, >140 bpm can be critical
    "Respiratory Rate": 35,          # Severe tachypnea, >30 is alarming
    "Body Temperature": 40.5,        # Hyperpyrexia, >40°C is dangerous
    "Oxygen Saturation": 85,         # Hypoxemia, <90% is critical
    "Systolic Blood Pressure": 180,  # Hypertensive crisis
    "Diastolic Blood Pressure": 120, # Hypertensive crisis
    "Age": 65,                        # Older age increases risk
    "Gender": np.nan,                      # Gender encoding same as your system
    "Weight (kg)": 120,               # Obesity-related risk
    "Height (m)": 1.75,               # Same as normal
    "Derived_HRV": 20,                # Very low HRV indicates poor autonomic function
    "Derived_Pulse_Pressure": 60,     # High PP indicates arterial stiffness
    "Derived_BMI": 39,                # Obesity class II-III
    "Derived_MAP": 130                 # Mean arterial pressure dangerously high
}



In [29]:
import pandas as pd

patient_input_df = pd.DataFrame([patient_input])  # keep keys as column names
y_pred = pipeline.predict(patient_input_df)
y_pred_proba = pipeline.predict_proba(patient_input_df)[:, 1]  # risk probability
y_pred, y_pred_proba


(array([1]), array([0.96847589]))

In [26]:
patient_input_df

,Heart Rate,Respiratory Rate,Body Temperature,Oxygen Saturation,Systolic Blood Pressure,Diastolic Blood Pressure,Age,Gender,Weight (kg),Height (m),Derived_HRV,Derived_Pulse_Pressure,Derived_BMI,Derived_MAP
0,150,35,40.5,85,180,120,65,NaN,120,1.75,20,60,39,130


In [27]:
input_array = np.array([[
    patient_input.get(col, np.nan)
    for col in FEATURE_ORDER
]])
input_array




array([[150.  ,  35.  ,  40.5 ,  85.  , 180.  , 120.  ,  65.  ,    nan,
        120.  ,   1.75,  20.  ,  60.  ,  39.  , 130.  ]])

In [ ]:
prediction = pipeline.predict(patient_input_df)

print("HIGH RISK" if prediction[0] == 1 else "LOW RISK")



HIGH RISK


/home/sangam/programming/python/project/myvenv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [31]:
list=X.columns.tolist()
list


['Heart Rate',
 'Respiratory Rate',
 'Body Temperature',
 'Oxygen Saturation',
 'Systolic Blood Pressure',
 'Diastolic Blood Pressure',
 'Age',
 'Gender',
 'Weight (kg)',
 'Height (m)',
 'Derived_HRV',
 'Derived_Pulse_Pressure',
 'Derived_BMI',
 'Derived_MAP']

In [32]:
for col in list:
    if col in FEATURE_ORDER:
        continue
    else:
        print(f"{col} is NOT in the model features.")


In [33]:
patient_input_df = pd.DataFrame([patient_input])

patient_input_df = patient_input_df[X_train.columns]
prediction = pipeline.predict(patient_input_df)
print("HIGH RISK" if prediction[0] == 1 else "LOW RISK")
y_pred_proba = pipeline.predict_proba(patient_input_df)[:,1]
print("Risk Probability:", y_pred_proba[0])


HIGH RISK
Risk Probability: 0.9684758906480869


In [35]:
joblib.dump(pipeline, "risk_predictor.pkl")

['risk_predictor.pkl']

In [ ]:
pipeline_loaded = joblib.load("risk_predictor.pkl")
pipeline_loaded.predict(patient_input_df)
\
if 

array([1])